# MAP Multi-LLM Submission Notebook

This notebook builds the final competition submission from the probability artifacts produced by `map_multi_llm_training.ipynb`.

Expected setup:

- attach the competition dataset
- attach one Kaggle dataset made from the exported `artifacts/` directory
- each run should live in its own folder under that artifact root

The ensemble follows the silver-writeup pattern:

- average weighted class probabilities
- add an agreement bonus based on top-K overlap
- add a max-confidence bonus
- family-filter the final ranking (`True_` vs `False_`)


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

print("Notebook ready.")


In [ ]:
# =========================
# EDIT THESE PATHS
# =========================
COMPETITION_DATA_DIR = Path("/kaggle/input/map-charting-student-math-misunderstandings")
ARTIFACT_ROOT = Path("/kaggle/input/map-multi-llm-artifacts")
OUTPUT_PATH = Path("/kaggle/working/submission.csv")

# Set to a list like ["qwen3_4b_lora", "qwen2_7b_qlora"] if you want a subset.
SELECT_RUN_SLUGS = None
TOP_N_BY_VALIDATION = None
MIN_VALIDATION_MAP3 = None
USE_VALIDATION_SCORE_WEIGHTS = True
VALIDATION_WEIGHT_POWER = 4.0

BASE_SCORE_WEIGHT = 0.34
AGREEMENT_WEIGHT = 0.33
CONFIDENCE_WEIGHT = 0.33
VOTE_TOP_K = 10


In [ ]:
# =========================
# HELPERS
# =========================
def discover_run_dirs(artifact_root: Path, selected_slugs=None):
    run_dirs = []
    for path in sorted(artifact_root.iterdir()):
        if path.is_dir() and (path / "metadata.json").exists() and (path / "test_probabilities.npz").exists():
            run_dirs.append(path)
    if selected_slugs:
        selected = set(selected_slugs)
        run_dirs = [path for path in run_dirs if path.name in selected]
    if not run_dirs:
        raise FileNotFoundError("No run folders were found under ARTIFACT_ROOT.")
    return run_dirs


def build_family_prefix(test_df: pd.DataFrame, correct_by_q: dict) -> pd.Series:
    correct_answer = test_df["QuestionId"].map(
        lambda qid: correct_by_q.get(int(qid)) if pd.notna(qid) else None
    )
    is_correct = np.where(
        correct_answer.notna(),
        (test_df["MC_Answer"].fillna("") == correct_answer).astype(int),
        pd.NA,
    )
    family_prefix = pd.Series(is_correct).map(
        lambda x: None if pd.isna(x) else ("True_" if int(x) == 1 else "False_")
    )
    return family_prefix


def select_top3(score_row, classes, family_prefix=None, family_fillers=None):
    ranked = sorted(zip(classes, score_row), key=lambda x: x[1], reverse=True)
    if family_prefix:
        ranked = [item for item in ranked if item[0].startswith(family_prefix)]
    selected = [label for label, _ in ranked[:3]]
    if family_prefix and family_fillers:
        for filler in family_fillers.get(family_prefix, []):
            if filler not in selected:
                selected.append(filler)
            if len(selected) == 3:
                break
    if len(selected) < 3:
        for label, _ in sorted(zip(classes, score_row), key=lambda x: x[1], reverse=True):
            if label not in selected:
                selected.append(label)
            if len(selected) == 3:
                break
    return selected[:3]


def reorder_probs_to_master(row_ids, probs, master_row_ids):
    if np.array_equal(row_ids, master_row_ids):
        return probs
    lookup = {int(row_id): idx for idx, row_id in enumerate(row_ids.tolist())}
    order = [lookup[int(row_id)] for row_id in master_row_ids.tolist()]
    return probs[order]


def align_classes_to_master(probs, source_classes, master_classes):
    if source_classes == master_classes:
        return probs
    source_index = {label: idx for idx, label in enumerate(source_classes)}
    take = [source_index[label] for label in master_classes]
    return probs[:, take]


In [ ]:
# =========================
# LOAD TEST + RUN METADATA
# =========================
test = pd.read_csv(COMPETITION_DATA_DIR / "test.csv")
run_dirs = discover_run_dirs(ARTIFACT_ROOT, SELECT_RUN_SLUGS)

print("Using run folders:")
for path in run_dirs:
    print(" -", path.name)

run_metadata = []
for path in run_dirs:
    metadata = json.loads((path / "metadata.json").read_text(encoding="utf-8"))
    metadata["run_dir"] = path
    run_metadata.append(metadata)

master_metadata = run_metadata[0]
master_classes = master_metadata["classes"]
family_fillers = master_metadata["family_fillers"]
correct_by_q = {int(k): v for k, v in master_metadata["correct_answer_map"].items()}
master_row_ids = test["row_id"].to_numpy()
family_prefixes = build_family_prefix(test, correct_by_q).tolist()

summary = pd.DataFrame(
    {
        "run_slug": [meta["run_slug"] for meta in run_metadata],
        "model_id": [meta["model_id"] for meta in run_metadata],
        "method": [meta["method"] for meta in run_metadata],
        "seed": [meta.get("seed") for meta in run_metadata],
        "val_map3_family": [meta.get("validation_map3_family") for meta in run_metadata],
        "ensemble_weight": [meta.get("ensemble_weight", 1.0) for meta in run_metadata],
    }
).sort_values("val_map3_family", ascending=False)

if MIN_VALIDATION_MAP3 is not None:
    keep = set(summary.loc[summary["val_map3_family"] >= MIN_VALIDATION_MAP3, "run_slug"])
    run_metadata = [meta for meta in run_metadata if meta["run_slug"] in keep]
    summary = summary[summary["run_slug"].isin(keep)].copy()

if TOP_N_BY_VALIDATION is not None:
    keep = set(summary.head(TOP_N_BY_VALIDATION)["run_slug"])
    run_metadata = [meta for meta in run_metadata if meta["run_slug"] in keep]
    summary = summary[summary["run_slug"].isin(keep)].copy()

if not run_metadata:
    raise ValueError("No runs remain after validation-score filtering.")

summary = summary.sort_values("val_map3_family", ascending=False).reset_index(drop=True)
summary


In [ ]:
# =========================
# LOAD PROBABILITY MATRICES
# =========================
loaded_runs = []

for meta in run_metadata:
    bundle = np.load(meta["run_dir"] / meta["probability_file"])
    probs = bundle["probabilities"].astype(np.float32)
    row_ids = bundle["row_id"]

    probs = reorder_probs_to_master(row_ids, probs, master_row_ids)
    probs = align_classes_to_master(probs, meta["classes"], master_classes)

    loaded_runs.append(
        {
            "run_slug": meta["run_slug"],
            "base_weight": float(meta.get("ensemble_weight", 1.0)),
            "val_map3_family": meta.get("validation_map3_family"),
            "probs": probs,
        }
    )

print("Loaded", len(loaded_runs), "probability matrices.")
print("Matrix shape:", loaded_runs[0]["probs"].shape)


In [ ]:
# =========================
# ENSEMBLE
# =========================
n_rows = len(master_row_ids)
n_classes = len(master_classes)

prob_sum = np.zeros((n_rows, n_classes), dtype=np.float32)
agreement = np.zeros((n_rows, n_classes), dtype=np.float32)
max_confidence = np.zeros((n_rows, n_classes), dtype=np.float32)

total_weight = 0.0
row_index = np.arange(n_rows)[:, None]

for run in loaded_runs:
    probs = run["probs"]
    weight = run["base_weight"]
    if USE_VALIDATION_SCORE_WEIGHTS and run["val_map3_family"] is not None:
        weight *= float(run["val_map3_family"]) ** VALIDATION_WEIGHT_POWER
    run["effective_weight"] = weight
    total_weight += weight

    prob_sum += probs * weight
    max_confidence = np.maximum(max_confidence, probs)

    k = min(VOTE_TOP_K, probs.shape[1])
    topk_idx = np.argpartition(-probs, kth=k - 1, axis=1)[:, :k]
    agreement[row_index, topk_idx] += weight

weight_table = pd.DataFrame(
    {
        "run_slug": [run["run_slug"] for run in loaded_runs],
        "val_map3_family": [run["val_map3_family"] for run in loaded_runs],
        "effective_weight": [run["effective_weight"] for run in loaded_runs],
    }
).sort_values("effective_weight", ascending=False)
weight_table

prob_component = prob_sum / max(total_weight, 1e-8)
agreement_component = agreement / max(total_weight, 1e-8)

final_scores = (
    prob_component * BASE_SCORE_WEIGHT
    + agreement_component * AGREEMENT_WEIGHT
    + max_confidence * CONFIDENCE_WEIGHT
)

top3_predictions = [
    select_top3(
        score_row=final_scores[i],
        classes=master_classes,
        family_prefix=family_prefixes[i],
        family_fillers=family_fillers,
    )
    for i in range(n_rows)
]

print("Example predictions:")
for preds in top3_predictions[:3]:
    print(preds)


In [ ]:
# =========================
# BUILD SUBMISSION
# =========================
submission = pd.DataFrame(
    {
        "Category:Misconception": [" ".join(preds) for preds in top3_predictions]
    }
)
submission.to_csv(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH)
submission.head()
